일단 기본 4개 노드 4개 원형 엣지 구성으로 감


In [ ]:
from qiskit import *


import torch

import numpy as np
import matplotlib.pyplot as plt

import random

from EGATE import *
# from NNVQE_HEA import *

In [ ]:
def generate_random_hamiltonian_graph(N=4, k=0, num_data=20):
    # 현재 RNG state(시드 상태) 백업
    # old_torch_state = torch.random.get_rng_state()
    # old_numpy_state = np.random.get_state()
    # old_py_state = random.getstate()

    # 그래프만 "진짜 랜덤"으로 만들기 위해, 시드를 None 또는 시스템 시계로 세팅
    # torch.manual_seed(int(time.time() * 1e6) + os.getpid())
    # np.random.seed(int(time.time()))
    # random.seed(None)
    A = np.linspace(-3.0,3.0,num_data)
    # 랜덤 그래프 생성 edges도 달라질 수 있어야함
    edges = [(i, i+1) for i in range(N-1)]
    edges.append((N-1, 0))
    edge_features = {}
    rand_vec = torch.rand(3)  # 기본적으로 [0,1] -> [0,2] -> [-1,1]
    
    # 첫 번째, 두 번째 값을 1로 지정
    rand_vec[0] = 1.0
    rand_vec[1] = 1.0
    rand_vec[2] = A[k]
    for e in edges:
        # edge_features[e] = torch.randint(-1, 2, (3,), dtype=torch.float32)  # 👈 0 또는 1로 설정
        edge_features[e] = rand_vec
        
    # 그래프 생성 완료 후, 복원
    # torch.random.set_rng_state(old_torch_state)
    # np.random.set_state(old_numpy_state)
    # random.setstate(old_py_state)

    return edges, edge_features

In [ ]:
# H-graph latent vector 구성
num_data = 20
latent_data = []
edge_full_data = []
if __name__ == "__main__":
    torch.manual_seed(1)
    np.random.seed(1)
    random.seed(1)

    for i in range(num_data):
        N = 8
        node_feats = one_hot_encoding_nodes(N)
        # print(node_feats)
        edges, edge_feats_dict = generate_random_hamiltonian_graph(N,i,num_data)
        # print(edge_feats_dict)
        edge_full_data.append(edge_feats_dict)
        # print("\n=== Training Graph AutoEncoder [EGAT Layer with λ splitting] ===",i)
    model = EGATEAutoEncoder(
            node_in_dim= N,
            edge_in_dim= 3,
            node_hidden_dim= N,  
            edge_hidden_dim= 3,
            decoder_hidden_dim = 55,
            num_layers=9,
            lambda_param=0.5,
            latent_dim = 11,
            num_edges=len(edges)
        )
    num_epochs = 101
    
    latent_data, mse_list, H_rec, E_rec, H_mse_list, E_mse_list = train_multi_graph(model, num_data, edges, edge_full_data, node_feats, num_epochs=num_epochs)
    torch.save(model.state_dict(), "gae_model.pth")
    for i, lat in enumerate(latent_data):
        # print(edge_full_data[i])
            print(f"Graph {i+1}: latent = {lat}")

In [ ]:
plt.plot(list(range(len(list(mse_list)))), list(mse_list))
plt.show()

In [ ]:
# H_mse_list = [t.detach().cpu().item() for t in H_mse_list]
plt.plot(list(range(len(list(H_mse_list)))), H_mse_list)
plt.show()

In [ ]:
# E_mse_list = [t.detach().cpu().item() for t in E_mse_list]
plt.plot(list(range(len(list(H_mse_list)))), E_mse_list)
plt.show()

In [ ]:
torch.save(edge_full_data, 'edge_full_data_train.pt')
torch.save(latent_data, 'latent_data_train.pt')

# 불러오기


In [ ]:
def generate_random_hamiltonian_graph_(N=4, k=0, num_data=20):

    A = np.linspace(-10.0,10.0,num_data)
    # 랜덤 그래프 생성 edges도 달라질 수 있어야함
    edges = [(i, i+1) for i in range(N-1)]
    edges.append((N-1, 0))
    edge_features = {}
    rand_vec = torch.rand(3)  # 기본적으로 [0,1] -> [0,2] -> [-1,1]
    
    # 첫 번째, 두 번째 값을 1로 지정
    rand_vec[0] = 1.0
    rand_vec[1] = 1.0
    rand_vec[2] = A[k]
    for e in edges:
        # edge_features[e] = torch.randint(-1, 2, (3,), dtype=torch.float32)  # 👈 0 또는 1로 설정
        edge_features[e] = rand_vec
        
    return edges, edge_features

In [ ]:
num_test_data = 1000
test_edge_full_data = []
if __name__ == "__main__":
        # torch.manual_seed(1103)
        # np.random.seed(1103)
        # random.seed(1103)

    for i in range(num_test_data):

        N = 8
        node_feats = one_hot_encoding_nodes(N)
        # print(node_feats)
        edges, edge_feats_dict = generate_random_hamiltonian_graph_(N,i,num_test_data)
        # print(edge_feats_dict)
        test_edge_full_data.append(edge_feats_dict)

In [ ]:
model = EGATEAutoEncoder(
            node_in_dim= N,
            edge_in_dim= 3,
            node_hidden_dim= N,  
            edge_hidden_dim= 3,
            decoder_hidden_dim = 55,
            num_layers=9,
            lambda_param=0.5,
            latent_dim = 11,
            num_edges=len(edges)
        )
model.load_state_dict(torch.load("gae_model.pth"))
model.eval()
test_latent = []
H_rec = []
E_rec = []
for i in range(num_test_data):
    edge_feats_dict= test_edge_full_data[i]
    E_in_list = [edge_feats_dict[e] for e in edges]
    E_in = torch.stack(E_in_list, dim=0)
    # E_in = torch.stack([edge_feats_dict[e] for e in edges])
    # print(E_in)
    with torch.no_grad():
        H_re, E_re, graph_latent = model(node_feats, E_in, edges)
        H_rec.append(H_re)
        E_rec.append(E_re)
    test_latent.append(graph_latent)
mse = nn.MSELoss()
mse_list = []
for i in range(num_test_data):
    edge_feats_dict= test_edge_full_data[i]
    E_in_list = [edge_feats_dict[e] for e in edges]
    E_in = torch.stack(E_in_list, dim=0)
    E_loss = mse(E_rec[i], E_in)
    H_loss = mse(H_rec[i], node_feats)
    mse_list.append(E_loss+H_loss)

In [ ]:
# E_mse_list = [t.detach().cpu().item() for t in E_mse_list]
plt.plot(list( np.linspace(-10.0,10.0,num_test_data)), mse_list)
print(np.mean(mse_list))
plt.show()

In [ ]:
torch.save(test_edge_full_data, 'edge_full_data_test.pt')
torch.save(test_latent, 'latent_data_test.pt')

# 불러오기
